# Coastal flood step 08: minimum vs maximum scenario comparison

Compares minimum and maximum coastal scenarios across:
- total avoided expected annual damages (EAD)
- percent of total damages avoided
- avoided damages by sector and subsector
- return period damage totals and percentages

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
fx_jmd_per_usd = 150.0  # 1 USD = 150 JMD
mangrove_attribution_buffer_m = 5000

comparison_output_dir = base_path / "dphil_paper_3/results_coastal_scenario_comparison"
comparison_output_dir.mkdir(parents=True, exist_ok=True)

scenario_paths = {
    "minimum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates",
    "maximum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates",
}

for scenario_name, scenario_path in scenario_paths.items():
    print(scenario_name, "->", scenario_path)
    if not scenario_path.exists():
        raise FileNotFoundError(f"Missing scenario output folder: {scenario_path}")



In [ ]:
def load_scenario_tables(damage_estimates_path: Path):
    sector_ead = pd.read_csv(damage_estimates_path / "coastal_ead_sector_summary_usd_with_pct_avoided.csv")
    subsector_ead = pd.read_csv(damage_estimates_path / "coastal_ead_subsector_summary_usd_with_pct_avoided.csv")
    rp_jd = pd.read_csv(damage_estimates_path / "sector_subsector_return_period_damages_with_avoided_share.csv")
    return sector_ead, subsector_ead, rp_jd


def load_attribution_tables(damage_estimates_path: Path, attribution_buffer_m: int):
    attribution_dir = damage_estimates_path / "mangrove_attribution"
    sector_attribution = pd.read_csv(
        attribution_dir / f"attribution_breakdown_by_sector_{attribution_buffer_m}m.csv"
    )
    subsector_attribution = pd.read_csv(
        attribution_dir / f"attribution_breakdown_by_subsector_{attribution_buffer_m}m.csv"
    )
    return sector_attribution, subsector_attribution


data = {}
attribution_data = {}
for scenario_name, scenario_path in scenario_paths.items():
    data[scenario_name] = load_scenario_tables(scenario_path)
    attribution_data[scenario_name] = load_attribution_tables(
        scenario_path,
        mangrove_attribution_buffer_m,
    )

print("Loaded scenario tables:", list(data.keys()))
print("Loaded attribution tables for buffer (m):", mangrove_attribution_buffer_m)



In [ ]:
def total_ead_summary(sector_ead_df: pd.DataFrame):
    total_with = sector_ead_df["EAD_With_Mangroves_USD"].sum()
    total_without = sector_ead_df["EAD_Without_Mangroves_USD"].sum()
    total_avoided = sector_ead_df["Avoided_EAD_USD"].sum()
    pct_avoided = (100.0 * total_avoided / total_without) if total_without else np.nan
    return {
        "EAD_With_Mangroves_USD": total_with,
        "EAD_Without_Mangroves_USD": total_without,
        "Avoided_EAD_USD": total_avoided,
        "Percent_Avoided_EAD_vs_NoMangroves": pct_avoided,
    }


def total_attribution_summary(
    sector_attribution_df: pd.DataFrame,
    total_without_mangroves_usd: float,
    total_avoided_usd: float,
):
    attributed_avoided = float(sector_attribution_df["Attributed_EAD_USD"].sum())
    unattributed_avoided = float(sector_attribution_df["Unattributed_EAD_USD"].sum())
    return {
        "Attributed_Avoided_EAD_USD": attributed_avoided,
        "Unattributed_Avoided_EAD_USD": unattributed_avoided,
        "Percent_Attributed_Avoided_vs_NoMangroves": (
            100.0 * attributed_avoided / total_without_mangroves_usd
            if total_without_mangroves_usd
            else np.nan
        ),
        "Percent_Unattributed_Avoided_vs_NoMangroves": (
            100.0 * unattributed_avoided / total_without_mangroves_usd
            if total_without_mangroves_usd
            else np.nan
        ),
        "Percent_Attributed_of_Total_Avoided": (
            100.0 * attributed_avoided / total_avoided_usd if total_avoided_usd else np.nan
        ),
    }


totals_rows = []
for scenario_name in data.keys():
    sector_summary = total_ead_summary(data[scenario_name][0])
    attribution_summary = total_attribution_summary(
        attribution_data[scenario_name][0],
        sector_summary["EAD_Without_Mangroves_USD"],
        sector_summary["Avoided_EAD_USD"],
    )
    totals_rows.append({"Scenario": scenario_name, **sector_summary, **attribution_summary})

totals_compare = pd.DataFrame(totals_rows).sort_values("Scenario").reset_index(drop=True)
totals_compare



In [ ]:
# Sector-level comparison
min_sector = data["minimum"][0].copy().set_index("Sector").add_prefix("Min_")
max_sector = data["maximum"][0].copy().set_index("Sector").add_prefix("Max_")

min_sector_attribution = attribution_data["minimum"][0].copy().set_index("Sector").add_prefix("Min_Attr_")
max_sector_attribution = attribution_data["maximum"][0].copy().set_index("Sector").add_prefix("Max_Attr_")

sector_compare = (
    min_sector
    .join(max_sector, how="outer")
    .join(min_sector_attribution, how="outer")
    .join(max_sector_attribution, how="outer")
    .reset_index()
)

sector_compare["Min_Attributed_Percent_vs_NoMangroves"] = np.where(
    sector_compare["Min_EAD_Without_Mangroves_USD"] > 0,
    100.0
    * sector_compare["Min_Attr_Attributed_EAD_USD"]
    / sector_compare["Min_EAD_Without_Mangroves_USD"],
    np.nan,
)
sector_compare["Max_Attributed_Percent_vs_NoMangroves"] = np.where(
    sector_compare["Max_EAD_Without_Mangroves_USD"] > 0,
    100.0
    * sector_compare["Max_Attr_Attributed_EAD_USD"]
    / sector_compare["Max_EAD_Without_Mangroves_USD"],
    np.nan,
)

sector_compare["Delta_Avoided_EAD_USD_MaxMinusMin"] = (
    sector_compare["Max_Avoided_EAD_USD"] - sector_compare["Min_Avoided_EAD_USD"]
)
sector_compare["Delta_Attributed_Avoided_EAD_USD_MaxMinusMin"] = (
    sector_compare["Max_Attr_Attributed_EAD_USD"] - sector_compare["Min_Attr_Attributed_EAD_USD"]
)

sector_compare



In [ ]:
# Subsector-level comparison
min_subsector = data["minimum"][1].copy().set_index(["Sector", "Subsector"]).add_prefix("Min_")
max_subsector = data["maximum"][1].copy().set_index(["Sector", "Subsector"]).add_prefix("Max_")

subsector_compare = min_subsector.join(max_subsector, how="outer").reset_index()
subsector_compare["Delta_Avoided_EAD_USD_MaxMinusMin"] = (
    subsector_compare["Max_Avoided_EAD_USD"] - subsector_compare["Min_Avoided_EAD_USD"]
)

subsector_compare

In [ ]:
def rp_total_summary(rp_jd_df: pd.DataFrame):
    grp = (
        rp_jd_df.groupby("ReturnPeriod", as_index=False)[
            ["Damages_With_Mangroves_JD", "Damages_Without_Mangroves_JD", "Avoided_Damages_JD"]
        ].sum()
    )
    grp["Damages_With_Mangroves_USD"] = grp["Damages_With_Mangroves_JD"] / fx_jmd_per_usd
    grp["Damages_Without_Mangroves_USD"] = grp["Damages_Without_Mangroves_JD"] / fx_jmd_per_usd
    grp["Avoided_Damages_USD"] = grp["Avoided_Damages_JD"] / fx_jmd_per_usd
    grp["Percent_Avoided_vs_NoMangroves"] = np.where(
        grp["Damages_Without_Mangroves_USD"] > 0,
        100.0 * grp["Avoided_Damages_USD"] / grp["Damages_Without_Mangroves_USD"],
        np.nan,
    )
    return grp


rp_min = rp_total_summary(data["minimum"][2]).add_prefix("Min_").rename(
    columns={"Min_ReturnPeriod": "ReturnPeriod"}
)
rp_max = rp_total_summary(data["maximum"][2]).add_prefix("Max_").rename(
    columns={"Max_ReturnPeriod": "ReturnPeriod"}
)

rp_compare = rp_min.merge(rp_max, on="ReturnPeriod", how="outer")
rp_compare

In [ ]:
totals_compare.to_csv(comparison_output_dir / "total_ead_comparison_minimum_vs_maximum.csv", index=False)
sector_compare.to_csv(comparison_output_dir / "sector_ead_comparison_minimum_vs_maximum.csv", index=False)
subsector_compare.to_csv(comparison_output_dir / "subsector_ead_comparison_minimum_vs_maximum.csv", index=False)
rp_compare.to_csv(comparison_output_dir / "return_period_damage_comparison_minimum_vs_maximum.csv", index=False)

for scenario_name, (sector_attribution_df, subsector_attribution_df) in attribution_data.items():
    sector_attribution_df.to_csv(
        comparison_output_dir
        / f"sector_attribution_breakdown_{scenario_name}_{mangrove_attribution_buffer_m}m.csv",
        index=False,
    )
    subsector_attribution_df.to_csv(
        comparison_output_dir
        / f"subsector_attribution_breakdown_{scenario_name}_{mangrove_attribution_buffer_m}m.csv",
        index=False,
    )

print("Saved comparison outputs to:", comparison_output_dir)
print("- total_ead_comparison_minimum_vs_maximum.csv")
print("- sector_ead_comparison_minimum_vs_maximum.csv")
print("- subsector_ead_comparison_minimum_vs_maximum.csv")
print("- return_period_damage_comparison_minimum_vs_maximum.csv")
print(f"- sector_attribution_breakdown_<scenario>_{mangrove_attribution_buffer_m}m.csv")
print(f"- subsector_attribution_breakdown_<scenario>_{mangrove_attribution_buffer_m}m.csv")



## Mangrove attribution area comparison

Compares mangrove area (ha and %) that has positive avoided EAD (damage reduction),
negative avoided EAD (damage increase), and zero attribution, for minimum and maximum scenarios.


In [ ]:
mangrove_source_candidates = [
    base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp",
    base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/mangroves_fn/mangroves.shp",
]
mangrove_source_path = next((path for path in mangrove_source_candidates if path.exists()), None)

if mangrove_source_path is not None:
    mangrove_source = gpd.read_file(mangrove_source_path).to_crs("EPSG:3448")
    total_mangrove_area_ha = float(mangrove_source.geometry.area.sum() / 10000.0)
    print("Total mangrove area source:", mangrove_source_path)
else:
    fallback_gpkg = (
        scenario_paths["minimum"]
        / "mangrove_attribution"
        / f"mangrove_attribution_total_{mangrove_attribution_buffer_m}m.gpkg"
    )
    mangrove_source = gpd.read_file(fallback_gpkg).to_crs("EPSG:3448")
    total_mangrove_area_ha = float(mangrove_source.geometry.area.sum() / 10000.0)
    print("Total mangrove area source fallback:", fallback_gpkg)

print(f"Total mangrove area (ha): {total_mangrove_area_ha:,.3f}")



In [ ]:
def summarize_mangrove_attribution_area(attribution_gpkg_path: Path, scenario_name: str):
    gdf = gpd.read_file(attribution_gpkg_path).to_crs("EPSG:3448").copy()
    value_col = "Total_Avoided_EAD_USD_attributed"
    if value_col not in gdf.columns:
        raise KeyError(f"Missing {value_col} in {attribution_gpkg_path}")

    gdf[value_col] = pd.to_numeric(gdf[value_col], errors="coerce").fillna(0.0)
    gdf["area_ha"] = gdf.geometry.area / 10000.0

    area_reduce = float(gdf.loc[gdf[value_col] > 0, "area_ha"].sum())
    area_increase = float(gdf.loc[gdf[value_col] < 0, "area_ha"].sum())
    area_zero = float(gdf.loc[gdf[value_col] == 0, "area_ha"].sum())

    count_reduce = int((gdf[value_col] > 0).sum())
    count_increase = int((gdf[value_col] < 0).sum())
    count_zero = int((gdf[value_col] == 0).sum())
    count_total = int(len(gdf))

    return {
        "Scenario": scenario_name,
        "Total_Mangrove_Area_ha": total_mangrove_area_ha,
        "Area_Reduces_Damage_ha": area_reduce,
        "Area_Increases_Damage_ha": area_increase,
        "Area_Zero_Attribution_ha": area_zero,
        "Pct_Area_Reduces_Damage_of_TotalMangroves": (100.0 * area_reduce / total_mangrove_area_ha) if total_mangrove_area_ha > 0 else np.nan,
        "Pct_Area_Increases_Damage_of_TotalMangroves": (100.0 * area_increase / total_mangrove_area_ha) if total_mangrove_area_ha > 0 else np.nan,
        "Pct_Area_Zero_Attribution_of_TotalMangroves": (100.0 * area_zero / total_mangrove_area_ha) if total_mangrove_area_ha > 0 else np.nan,
        "Count_Reduces_Damage": count_reduce,
        "Count_Increases_Damage": count_increase,
        "Count_Zero_Attribution": count_zero,
        "Count_Total_Mangroves": count_total,
    }


mangrove_area_rows = []
for scenario_name, scenario_damage_estimates_path in scenario_paths.items():
    attribution_gpkg = (
        scenario_damage_estimates_path
        / "mangrove_attribution"
        / f"mangrove_attribution_total_{mangrove_attribution_buffer_m}m.gpkg"
    )
    if not attribution_gpkg.exists():
        raise FileNotFoundError(f"Missing mangrove attribution gpkg: {attribution_gpkg}")
    mangrove_area_rows.append(summarize_mangrove_attribution_area(attribution_gpkg, scenario_name))

mangrove_area_compare = pd.DataFrame(mangrove_area_rows).sort_values("Scenario").reset_index(drop=True)
mangrove_area_compare



In [ ]:
mangrove_area_compare.to_csv(
    comparison_output_dir / "mangrove_attribution_area_comparison_minimum_vs_maximum.csv",
    index=False,
)
print("Saved:", comparison_output_dir / "mangrove_attribution_area_comparison_minimum_vs_maximum.csv")



## Total EAD bar charts

Readable bar charts for total EAD metrics by scenario (USD, non-scientific).


In [ ]:
if "totals_compare" not in globals():
    raise ValueError("Run the totals_compare cell first.")


def format_usd_millions(value, _position):
    return f"${value:,.1f}M"


chart_df = totals_compare.copy()

money_columns = [
    "EAD_With_Mangroves_USD",
    "EAD_Without_Mangroves_USD",
    "Avoided_EAD_USD",
    "Attributed_Avoided_EAD_USD",
]
for column_name in money_columns:
    chart_df[f"{column_name}_mn"] = chart_df[column_name] / 1e6

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

x_positions = np.arange(len(chart_df))
bar_width = 0.22

# Panel 1: total EAD components (includes total avoided)
axes[0].bar(x_positions - bar_width, chart_df["EAD_Without_Mangroves_USD_mn"], width=bar_width, label="Without mangroves", color="#4C78A8")
axes[0].bar(x_positions, chart_df["EAD_With_Mangroves_USD_mn"], width=bar_width, label="With mangroves", color="#F58518")
axes[0].bar(x_positions + bar_width, chart_df["Avoided_EAD_USD_mn"], width=bar_width, label="Avoided (total)", color="#2a9d8f")
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(chart_df["Scenario"])
axes[0].set_title("Total EAD comparison (USD millions)")
axes[0].set_ylabel("USD (millions)")
axes[0].yaxis.set_major_formatter(FuncFormatter(format_usd_millions))
axes[0].legend(frameon=False)

# Panel 2: avoided EAD total vs 5000m attributed
avoid_bar_width = 0.34
axes[1].bar(
    x_positions - avoid_bar_width / 2,
    chart_df["Avoided_EAD_USD_mn"],
    width=avoid_bar_width,
    label="Avoided (total)",
    color="#2a9d8f",
)
axes[1].bar(
    x_positions + avoid_bar_width / 2,
    chart_df["Attributed_Avoided_EAD_USD_mn"],
    width=avoid_bar_width,
    label=f"Avoided ({mangrove_attribution_buffer_m}m attributed)",
    color="#1b7837",
)
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(chart_df["Scenario"])
axes[1].set_title(f"Avoided EAD: total vs {mangrove_attribution_buffer_m}m attributed")
axes[1].set_ylabel("USD (millions)")
axes[1].yaxis.set_major_formatter(FuncFormatter(format_usd_millions))
axes[1].legend(frameon=False)

# Panel 3: percent avoided total vs 5000m attributed
pct_bar_width = 0.34
axes[2].bar(
    x_positions - pct_bar_width / 2,
    chart_df["Percent_Avoided_EAD_vs_NoMangroves"],
    width=pct_bar_width,
    label="Percent avoided (total)",
    color="#74C476",
)
axes[2].bar(
    x_positions + pct_bar_width / 2,
    chart_df["Percent_Attributed_Avoided_vs_NoMangroves"],
    width=pct_bar_width,
    label=f"Percent avoided ({mangrove_attribution_buffer_m}m attributed)",
    color="#238B45",
)
axes[2].set_xticks(x_positions)
axes[2].set_xticklabels(chart_df["Scenario"])
axes[2].set_title("Percent avoided vs no-mangrove baseline")
axes[2].set_ylabel("Percent (%)")
axes[2].set_ylim(
    0,
    max(
        1,
        chart_df[
            ["Percent_Avoided_EAD_vs_NoMangroves", "Percent_Attributed_Avoided_vs_NoMangroves"]
        ]
        .max()
        .max()
        * 1.25,
    ),
)
axes[2].legend(frameon=False)

for index, value in enumerate(chart_df["Percent_Avoided_EAD_vs_NoMangroves"]):
    axes[2].text(index - pct_bar_width / 2, value + 0.15, f"{value:.2f}%", ha="center", va="bottom", fontsize=8)
for index, value in enumerate(chart_df["Percent_Attributed_Avoided_vs_NoMangroves"]):
    axes[2].text(index + pct_bar_width / 2, value + 0.15, f"{value:.2f}%", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
out_png = comparison_output_dir / "totals_compare_bar_charts_total_vs_attributed_5000m.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_png)



## Sector comparison bar charts

Readable sector-level scenario comparison using USD millions and percent avoided.


In [ ]:
if "sector_compare" not in globals():
    raise ValueError("Run the sector_compare cell first.")

plot_df = sector_compare.copy()
plot_df = plot_df[plot_df["Sector"].notna()].copy().sort_values("Sector").reset_index(drop=True)

plot_df["Min_Avoided_EAD_USD_mn"] = plot_df["Min_Avoided_EAD_USD"] / 1e6
plot_df["Max_Avoided_EAD_USD_mn"] = plot_df["Max_Avoided_EAD_USD"] / 1e6
plot_df["Min_Attr_Attributed_EAD_USD_mn"] = plot_df["Min_Attr_Attributed_EAD_USD"] / 1e6
plot_df["Max_Attr_Attributed_EAD_USD_mn"] = plot_df["Max_Attr_Attributed_EAD_USD"] / 1e6

x_positions = np.arange(len(plot_df))
bar_width = 0.36

fig, axes = plt.subplots(2, 2, figsize=(18, 11))

# Top-left: sector avoided EAD total
axes[0, 0].bar(x_positions - bar_width / 2, plot_df["Min_Avoided_EAD_USD_mn"], width=bar_width, label="Minimum scenario", color="#2a9d8f")
axes[0, 0].bar(x_positions + bar_width / 2, plot_df["Max_Avoided_EAD_USD_mn"], width=bar_width, label="Maximum scenario", color="#66c2a4")
axes[0, 0].set_xticks(x_positions)
axes[0, 0].set_xticklabels(plot_df["Sector"], rotation=30, ha="right")
axes[0, 0].set_title("Sector avoided EAD (total)")
axes[0, 0].set_ylabel("Avoided EAD (USD millions)")
axes[0, 0].yaxis.set_major_formatter(FuncFormatter(format_usd_millions))
axes[0, 0].legend(frameon=False)

# Top-right: sector percent avoided total
axes[0, 1].bar(x_positions - bar_width / 2, plot_df["Min_Percent_Avoided_EAD_vs_NoMangroves"], width=bar_width, label="Minimum scenario", color="#74C476")
axes[0, 1].bar(x_positions + bar_width / 2, plot_df["Max_Percent_Avoided_EAD_vs_NoMangroves"], width=bar_width, label="Maximum scenario", color="#238B45")
axes[0, 1].set_xticks(x_positions)
axes[0, 1].set_xticklabels(plot_df["Sector"], rotation=30, ha="right")
axes[0, 1].set_title("Sector percent avoided (total)")
axes[0, 1].set_ylabel("Percent avoided (%)")
axes[0, 1].legend(frameon=False)

# Bottom-left: sector avoided EAD attributed only
axes[1, 0].bar(x_positions - bar_width / 2, plot_df["Min_Attr_Attributed_EAD_USD_mn"], width=bar_width, label="Minimum scenario", color="#1b7837")
axes[1, 0].bar(x_positions + bar_width / 2, plot_df["Max_Attr_Attributed_EAD_USD_mn"], width=bar_width, label="Maximum scenario", color="#5aae61")
axes[1, 0].set_xticks(x_positions)
axes[1, 0].set_xticklabels(plot_df["Sector"], rotation=30, ha="right")
axes[1, 0].set_title(f"Sector avoided EAD ({mangrove_attribution_buffer_m}m attributed)")
axes[1, 0].set_ylabel("Avoided EAD (USD millions)")
axes[1, 0].yaxis.set_major_formatter(FuncFormatter(format_usd_millions))
axes[1, 0].legend(frameon=False)

# Bottom-right: sector percent avoided attributed only
axes[1, 1].bar(x_positions - bar_width / 2, plot_df["Min_Attributed_Percent_vs_NoMangroves"], width=bar_width, label="Minimum scenario", color="#2E8B57")
axes[1, 1].bar(x_positions + bar_width / 2, plot_df["Max_Attributed_Percent_vs_NoMangroves"], width=bar_width, label="Maximum scenario", color="#006D2C")
axes[1, 1].set_xticks(x_positions)
axes[1, 1].set_xticklabels(plot_df["Sector"], rotation=30, ha="right")
axes[1, 1].set_title(f"Sector percent avoided ({mangrove_attribution_buffer_m}m attributed)")
axes[1, 1].set_ylabel("Percent avoided (%)")
axes[1, 1].legend(frameon=False)

plt.tight_layout()
out_png = comparison_output_dir / "sector_compare_bar_charts_total_vs_attributed_5000m.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_png)



## Sector full totals charts (with, without, avoided)

Shows sector totals for each scenario with three bars per sector:
- damages without mangroves
- damages with mangroves
- avoided damages


In [ ]:
if "sector_compare" not in globals():
    raise ValueError("Run the sector_compare cell first.")

plot_df = sector_compare.copy()
plot_df = plot_df[plot_df["Sector"].notna()].copy()
plot_df = plot_df.sort_values("Sector").reset_index(drop=True)

for prefix in ["Min", "Max"]:
    plot_df[f"{prefix}_EAD_Without_Mangroves_USD_mn"] = plot_df[f"{prefix}_EAD_Without_Mangroves_USD"] / 1e6
    plot_df[f"{prefix}_EAD_With_Mangroves_USD_mn"] = plot_df[f"{prefix}_EAD_With_Mangroves_USD"] / 1e6
    plot_df[f"{prefix}_Avoided_EAD_USD_mn"] = plot_df[f"{prefix}_Avoided_EAD_USD"] / 1e6
    plot_df[f"{prefix}_Attr_Attributed_EAD_USD_mn"] = plot_df[f"{prefix}_Attr_Attributed_EAD_USD"] / 1e6

x_positions = np.arange(len(plot_df))
bar_width = 0.18

fig, axes = plt.subplots(1, 2, figsize=(19, 6), sharey=True)

# Minimum scenario
axes[0].bar(x_positions - 1.5 * bar_width, plot_df["Min_EAD_Without_Mangroves_USD_mn"], width=bar_width, label="Without mangroves", color="#4C78A8")
axes[0].bar(x_positions - 0.5 * bar_width, plot_df["Min_EAD_With_Mangroves_USD_mn"], width=bar_width, label="With mangroves", color="#F58518")
axes[0].bar(x_positions + 0.5 * bar_width, plot_df["Min_Avoided_EAD_USD_mn"], width=bar_width, label="Avoided (total)", color="#2a9d8f")
axes[0].bar(
    x_positions + 1.5 * bar_width,
    plot_df["Min_Attr_Attributed_EAD_USD_mn"],
    width=bar_width,
    label=f"Avoided ({mangrove_attribution_buffer_m}m attributed)",
    color="#1b7837",
)
axes[0].set_title("Minimum scenario: sector EAD totals")
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(plot_df["Sector"], rotation=30, ha="right")
axes[0].set_ylabel("USD (millions)")
axes[0].yaxis.set_major_formatter(FuncFormatter(format_usd_millions))
axes[0].legend(frameon=False)

# Maximum scenario
axes[1].bar(x_positions - 1.5 * bar_width, plot_df["Max_EAD_Without_Mangroves_USD_mn"], width=bar_width, label="Without mangroves", color="#4C78A8")
axes[1].bar(x_positions - 0.5 * bar_width, plot_df["Max_EAD_With_Mangroves_USD_mn"], width=bar_width, label="With mangroves", color="#F58518")
axes[1].bar(x_positions + 0.5 * bar_width, plot_df["Max_Avoided_EAD_USD_mn"], width=bar_width, label="Avoided (total)", color="#2a9d8f")
axes[1].bar(
    x_positions + 1.5 * bar_width,
    plot_df["Max_Attr_Attributed_EAD_USD_mn"],
    width=bar_width,
    label=f"Avoided ({mangrove_attribution_buffer_m}m attributed)",
    color="#1b7837",
)
axes[1].set_title("Maximum scenario: sector EAD totals")
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(plot_df["Sector"], rotation=30, ha="right")
axes[1].yaxis.set_major_formatter(FuncFormatter(format_usd_millions))
axes[1].legend(frameon=False)

plt.tight_layout()
out_png = comparison_output_dir / "sector_compare_full_totals_with_total_and_attributed_avoided_5000m.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_png)



## Sector percent avoided by scenario

Companion chart for sector totals, showing percent avoided for minimum and maximum scenarios by sector.


In [ ]:
if "sector_compare" not in globals():
    raise ValueError("Run the sector_compare cell first.")

pct_df = sector_compare.copy()
pct_df = pct_df[pct_df["Sector"].notna()].copy().sort_values("Sector").reset_index(drop=True)

x_positions = np.arange(len(pct_df))
bar_width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

# Total percent avoided
axes[0].bar(x_positions - bar_width / 2, pct_df["Min_Percent_Avoided_EAD_vs_NoMangroves"], width=bar_width, label="Minimum scenario", color="#74C476")
axes[0].bar(x_positions + bar_width / 2, pct_df["Max_Percent_Avoided_EAD_vs_NoMangroves"], width=bar_width, label="Maximum scenario", color="#238B45")
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(pct_df["Sector"], rotation=30, ha="right")
axes[0].set_ylabel("Percent avoided (%)")
axes[0].set_title("Sector percent avoided (total)")
axes[0].legend(frameon=False)

# 5000m attributed-only percent avoided
axes[1].bar(x_positions - bar_width / 2, pct_df["Min_Attributed_Percent_vs_NoMangroves"], width=bar_width, label="Minimum scenario", color="#2E8B57")
axes[1].bar(x_positions + bar_width / 2, pct_df["Max_Attributed_Percent_vs_NoMangroves"], width=bar_width, label="Maximum scenario", color="#006D2C")
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(pct_df["Sector"], rotation=30, ha="right")
axes[1].set_title(f"Sector percent avoided ({mangrove_attribution_buffer_m}m attributed)")
axes[1].legend(frameon=False)

all_pct_values = np.concatenate([
    pct_df["Min_Percent_Avoided_EAD_vs_NoMangroves"].to_numpy(dtype=float),
    pct_df["Max_Percent_Avoided_EAD_vs_NoMangroves"].to_numpy(dtype=float),
    pct_df["Min_Attributed_Percent_vs_NoMangroves"].to_numpy(dtype=float),
    pct_df["Max_Attributed_Percent_vs_NoMangroves"].to_numpy(dtype=float),
])
finite_pct_values = all_pct_values[np.isfinite(all_pct_values)]
if finite_pct_values.size > 0:
    shared_pct_max = max(1.0, float(finite_pct_values.max()) * 1.2)
    axes[0].set_ylim(0, shared_pct_max)
    axes[1].set_ylim(0, shared_pct_max)

plt.tight_layout()
out_png = comparison_output_dir / "sector_percent_avoided_total_vs_attributed_5000m.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_png)

